# 07 · Using a trained model on any image

Everything so far scored stored scenes. This notebook runs the model on **image files**, the way you would
use it for real:

1. **Open** the file and make it a scene: crop the middle square, resize it to `SIZE`×`SIZE` (`to_scene`).
2. **Predict**: one forward pass gives a class, where the heatmap peaks, and how high it peaks (`predict`).
3. **Decide**: if the peak's probability (its *presence*) is at least τ, answer "sign: <class>"; otherwise
   "no sign". τ is the cutoff chosen in notebook 06 so that 1% of empty scenes cross it.

Step 3 is why the default model here is `peak_decoy` (v3a.1): it was trained to keep its heatmap low when
there's no sign, so its presence means something. An earlier model (v1–v2) still names a class with high
confidence on an empty street, because every scene it ever saw had a sign.

**You'll see:** real sign windows and real empty windows run through the model, the same photo at two sizes,
and where to put your own images.

**Needs:** a trained run with a `presence_val.json` (notebook 06), and the GTSDB photos for the demo images.
**Writes:** demo images and overlays under `outputs/notebooks/` only.

In [ ]:
# ---- Settings: the only cell you need to edit ----
RUN_NAME = "peak_decoy"  # which trained run to use; it needs outputs/eval/<run>/presence_val.json for τ
MY_IMAGES = []  # your own image files (jpg, png, ...), e.g. ["~/Pictures/street.jpg"]; paths relative to the repo root also work
SIZE = 128  # scene side in px, a multiple of 4. 128 is what the model trained on; try 256 or 512 for big photos

In [ ]:
# Setup (the same in every notebook). All paths in this project (configs/, data/, outputs/) are relative
# to the repo root, and the scripts expect to run from there, so move this notebook's working folder up
# to the root first. Safe to run twice.
import os
import sys
from pathlib import Path

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists():  # walk up until we reach the folder holding pyproject.toml
    if root == root.parent:
        raise RuntimeError("can't find the repo root: open this notebook from inside FindTheObject-CNN-Project")
    root = root.parent
os.chdir(root)
if str(root / "notebooks") not in sys.path:
    sys.path.insert(0, str(root / "notebooks"))  # lets us import notebooks/nbhelpers.py

from nbhelpers import NB_OUT, ROOT, find_run, read_json, require, run_script
print("working in", ROOT)

In [ ]:
# Stop here, with a clear message, if a file this notebook reads is missing; then load the model and its τ.
from findobject.training.checkpoint import load_model
from findobject.utils.device import pick_device

GTSDB = ROOT / "data/raw/gtsdb/FullIJCNN2013"
require(GTSDB / "gt.txt", how="download GTSDB as described in README.md (Data section)")
require("data/processed/real_dev.npz", how="python scripts/make_real_dataset.py")
require("data/processed/real_empty_dev.npz", how="python scripts/make_empty_dataset.py --config configs/peak_fresh_aug.yaml")
RUN_DIR = find_run(RUN_NAME)
CKPT = (RUN_DIR / "best.pt").relative_to(ROOT)
tau_file = ROOT / "outputs/eval" / RUN_DIR.name / "presence_val.json"
require(tau_file, how=f"python scripts/evaluate_presence.py --checkpoint {CKPT} --split val")
TAU = read_json(tau_file)["tau"]

device = pick_device("auto")
model = load_model(CKPT, device)  # load_model returns the model already in eval mode
print(f"run {RUN_DIR.name}, τ = {TAU:.3f}, device {device}")

## 1. Make some demo image files

To show the model working on plain files, we write a few out first, all as ordinary images in
`outputs/notebooks/demo_images/`:

* 4 **real sign windows** from real_dev (128×128 crops of GTSDB road photos, each with one labeled sign),
* 4 **real empty windows** from real_empty_dev (crops of the same kind of photos with no labeled sign),
* 2 whole **GTSDB photos** (1360×800), copied as they are.

We keep each window's true box and class on the side, so the pictures can show the right answer too.

In [ ]:
# Write the demo images. Rewriting them each run is harmless: the same input gives the same files.
import shutil

import numpy as np
from PIL import Image

DEMO = NB_OUT / "demo_images"
DEMO.mkdir(parents=True, exist_ok=True)
truth = {}  # file name -> (box, class) for real sign windows; the model never sees this
with np.load(ROOT / "data/processed/real_dev.npz") as z:
    for i in range(4):
        name = f"real_sign_{i}.png"
        Image.fromarray(z["images"][i]).save(DEMO / name)
        truth[name] = (tuple(z["boxes"][i].tolist()), int(z["class_ids"][i]))
with np.load(ROOT / "data/processed/real_empty_dev.npz") as z:
    for i in range(4):
        Image.fromarray(z["images"][i]).save(DEMO / f"real_empty_{i}.png")
for photo in ("00000.ppm", "00017.ppm"):
    shutil.copyfile(GTSDB / photo, DEMO / f"gtsdb_{photo}")
demo_files = sorted(DEMO.iterdir())
for p in demo_files:
    with Image.open(p) as img:
        print(f"{p.relative_to(ROOT)}  {img.size[0]}x{img.size[1]}")

## 2. File → scene → prediction → answer

`look_at(path)` below is the whole pipeline in five lines. A `Prediction` holds:

* `class_id`, `class_prob`: the most likely of the 43 classes, and its softmax probability,
* `peak_xy`: the scene pixel at the center of the heatmap's brightest cell,
* `peak_prob`: that cell's probability, which is exactly notebook 06's *presence*,
* `heatmap`: the whole `[SIZE/4, SIZE/4]` grid of probabilities, for drawing.

In [ ]:
# The pipeline: open an image file, turn it into a scene, predict, and decide "sign" vs "no sign" with τ.
from findobject.data.class_names import CLASS_NAMES
from findobject.inference.predict import predict
from findobject.inference.preprocess import to_scene


def look_at(path, size=SIZE):
    """Return (scene, prediction, answer) for one image file."""
    with Image.open(path) as img:
        scene = to_scene(img, size)  # center square crop, resized to size x size, float [3, size, size] in 0..1
    pred = predict(model, scene[None].to(device))[0]  # [None] makes a batch of one
    answer = f"sign: {CLASS_NAMES[pred.class_id]}" if pred.peak_prob >= TAU else "no sign"
    return scene, pred, answer


for p in demo_files:
    _, pred, answer = look_at(p)
    print(f"{p.name:22} presence {pred.peak_prob:.2f} -> {answer:32} (top class {CLASS_NAMES[pred.class_id]}, {pred.class_prob:.0%})")

In [ ]:
# Draw each demo image as the model saw it: heatmap glow, cyan ring at the peak, and the answer.
# For real sign windows the true box is green, and a wrong class turns the title red.
import matplotlib.pyplot as plt

from findobject.viz.overlay import draw_prediction

fig, axes = plt.subplots(2, 5, figsize=(16, 7.4))
for ax, p in zip(axes.flat, demo_files):
    scene, pred, answer = look_at(p)
    box, cls = truth.get(p.name, (None, None))
    draw_prediction(ax, scene, pred, true_box=box, true_class=cls)
    ax.set_title(f"{p.name}\n{answer} (presence {pred.peak_prob:.2f})\n" + ax.get_title(), fontsize=8,
                 color=ax.title.get_color())
plt.tight_layout()
plt.show()

Two things to notice:

* On the empty windows the model **still names a class**: its class head always picks one of 43. Only the
  presence check turns that into "no sign". This is why τ matters.
* The two **whole photos** go differently. 00017's ice/snow sign is big (123 px wide in the photo), so it's
  found. 00000's right-of-way sign is 41 px wide, and the model answers "no sign". The next section shows why.

## 3. Big photos: size matters

The model learned signs that are 16–48 px across in a 128 px scene. `to_scene` squeezes a 1360×800 photo's
middle 800×800 square down to `SIZE`. At 128 that shrinks everything 6.25 times, so a 42 px sign becomes about
7 px, far smaller than anything the model trained on.

The network is **fully convolutional** (notebook 02), so the same weights run on a 512×512 scene. That shrinks
the photo only about 1.6 times, and the sign stays about 27 px: back in the range the model knows.

In [ ]:
# The same GTSDB photo at SIZE 128 and 512, with its labeled sign's box scaled into each scene.
from findobject.data.gtsdb import read_gtsdb

photo = read_gtsdb(GTSDB)[0]  # photo 00000.ppm and its labeled signs
fig, axes = plt.subplots(1, 2, figsize=(12, 6.4))
for ax, size in zip(axes, (128, 512)):
    scene, pred, answer = look_at(photo.path, size)
    scale, left = size / 800, (1360 - 800) // 2  # to_scene keeps the middle 800x800 square, then resizes it
    boxes = [tuple(round(v) for v in ((x1 - left) * scale, y1 * scale, (x2 - left) * scale, y2 * scale))
             for (x1, y1, x2, y2) in (s.box for s in photo.signs)]
    draw_prediction(ax, scene, pred, true_box=boxes[0], true_class=photo.signs[0].class_id)
    side = boxes[0][2] - boxes[0][0]
    ax.set_title(f"SIZE {size}: sign is {side} px wide; {answer} (presence {pred.peak_prob:.2f})\n" + ax.get_title(),
                 fontsize=9, color=ax.title.get_color())
plt.tight_layout()
plt.show()

A caution about using a bigger `SIZE`: τ was chosen on 128×128 scenes, which have 1,024 heatmap cells. A
512×512 scene has 16,384, and presence is the *highest* of them. More cells means more chances for one to fire
by accident, so on big scenes the false-alarm rate is likely above the 1% τ was set for. This hasn't been
measured.

## 4. Your own images

Put file paths in `MY_IMAGES` in the Settings cell and re-run from the top. Any image format Pillow can open
works. Signs work best when they fill roughly 1/8 to 1/3 of the image's shorter side, because that's the size
range the model trained on. If your sign is small in a big photo, crop closer or raise `SIZE`.

In [ ]:
# Run the model on your own images (MY_IMAGES in the Settings cell).
from pathlib import Path

if not MY_IMAGES:
    print("MY_IMAGES is empty: add some image paths in the Settings cell and re-run this notebook.")
else:
    paths = [Path(p).expanduser() for p in MY_IMAGES]
    paths = [p if p.is_absolute() else ROOT / p for p in paths]
    fig, axes = plt.subplots(1, len(paths), figsize=(4 * len(paths), 4.6), squeeze=False)
    for ax, p in zip(axes[0], paths):
        scene, pred, answer = look_at(p)
        draw_prediction(ax, scene, pred)
        ax.set_title(f"{p.name}\n{answer} (presence {pred.peak_prob:.2f})", fontsize=9)
    plt.tight_layout()
    plt.show()

## The script version

`scripts/infer.py` does the same pipeline from the command line and saves one overlay picture per image.
It prints `peak_prob` but doesn't apply τ, so you compare it with τ yourself. This cell always runs: it only
writes into `outputs/notebooks/infer/`.

In [ ]:
# Run scripts/infer.py on the demo images; it writes <name>_overlay.png for each into outputs/notebooks/infer/.
INFER_OUT = (NB_OUT / "infer").relative_to(ROOT)
run_script("infer", "--checkpoint", CKPT, "--size", SIZE, "--out", INFER_OUT,
           *[p.relative_to(ROOT) for p in demo_files])
print(f"compare each peak_prob with τ = {TAU:.3f}")

**Next:** `08_diagnostics.ipynb`: the project's one-off checks (data pictures, shortcut tests, memorization
probes), runnable from one place.